# Solve Field

This notebook can be used to determine the WCS solution of an OmegaCAM image.
It uses astrometry.net solve_field command. It is a necessary step before running scamp which needs a WCS accurate to approx 1", otherwise it fails.

Before start please be sure you can run the solve_field command from the astrometry package by tipying
/usr/local/astrometry/bin/solve-field

If you get something like
ERROR: You didn't specify any files to process.<br>
This program is part of the Astrometry.net suite.<br>
For details, visit http://astrometry.net.<br>
Git URL https://github.com/dstndstn/astrometry.net<br>
Revision 0.89, date Mon_Jan_17_07:47:26_2022_-0500.<br>

then you are okay.

This notebook will create a independent fits image for each CCD within the input mosaic. Then run solve_field on each one to find an astrometric solution. This solution will be incorporated to a new mosaic which will be use later.
Please be sure you have enough space in your hard disk, as this notebook will effectively created two new mosaic images for each input one.

Tested on my personal laptop using the index fields 4100 and 5200 series from astromatic.
Known issues:
- For some reason, the astrometry package failed during the flow of the script, but running the same command in terminal directly over the CCD fits image retrieves a solution in seconds.
- For those CCDs that have failed at retrieve a solution with astrometry, are copied directly into the final mosaic, causing some shifts between CCDs.

Lientur

In [1]:
import os
import sys

import glob
import logging
import multiprocessing

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_solve
from pycavi import pycavi_tools
from pycavi import pycavi_instrument

This will be executed when the package is imported.


In [2]:
import multiprocessing

multiprocessing.cpu_count()

8

In [3]:
# Set the reduced science dir where it will look for the reduced science images, same as previous notebooks
# For each target within the folder will check the filters before start to work on each observation.
# In this case, during the tests I consider only the images in the filter NB_659.

config_dir = os.getcwd()+'/'+'../config/'
solve_field_path = "/usr/local/opt/astrometry-net/bin/solve-field"

config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
reduced_science_dir = config['Directories']['reduced_science_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file. This ensure
# proper keyword parsing.
#
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

# a list with the tags of image filters. During the tests I worked only with r_SDSS and NB_659
filters_to_sol = ["BB_U_38_ESO841", "BB_V_89_ESO843", "BB_B_99_ESO842", "g_SDSS", "r_SDSS", "NB_659", "NB_OIII_8_ESO859"]

# target list will include all the observed targets within the reduced_science directory
target_list = []
for object_ in glob.glob(reduced_science_dir + "*"):
    target_list.append(object_)

for object_ in glob.glob(reduced_std_dir + "*"):
    target_list.append(object_)
    
target_list.sort()

# for each observed target will all the images which filters correspond to the ones given in filters_to_sol
for target_ in target_list:
    for filter_ in filters_to_sol:
        for observation_ in glob.glob(target_ + "/" + filter_ + "/*"):
                print(observation_)
                pycavi_solve.run_py4_solve(observation_, instrument, 
                                           cpu_count=multiprocessing.cpu_count()-2, 
                                           redo_astrometry=False, 
                                           solve_field_path=solve_field_path)

# job done
logging.info("\n\n ****** Py4 Done! ******\n\n")

2024-08-23 15:23:54,794:INFO:current directory: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42
2024-08-23 15:23:54,797:INFO:solving image 01/03: WFI.2002-03-11T00:46:20.918_pre.fits
2024-08-23 15:23:54,798:INFO:


Solving field WFI.2002-03-11T00:46:20.918_pre
2024-08-23 15:23:54,800:INFO:run_solve_field: directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre created 
2024-08-23 15:23:54,809:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP1.OUT1_solve_field/ Created 
2024-08-23 15:23:54,973:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP2.OUT1_solve_field/ Created 


config_dir: /Users/fselman/Software/testing/Pycavi/notebooks/../config/ instrument: wfi
/Users/fselman/Pictures/WFI/30Doradus//science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42
WFI.*_pre.fits


2024-08-23 15:23:55,094:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP3.OUT1_solve_field/ Created 
2024-08-23 15:23:55,238:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP4.OUT1_solve_field/ Created 
2024-08-23 15:23:55,362:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP8.OUT1_solve_field/ Created 
2024-08-23 15:23:55,444:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP7.OUT1_solve_field/ Created 
2024-08-23 15:23:55,572:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Do

This will be executed when the package is imported.



****************** solvefield ***************
dir_name2: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP4.OUT1_solve_field  
file_name2: WIN1.CHIP4.OUT1.fits 
sub_dir_name: WIN1.CHIP4.OUT1_solve_field
solvefield: running solve-field in directory: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WIN1.CHIP4.OUT1_solve_field
******************



solvefield: command: /usr/local/opt/astrometry-net/bin/solve-field  --ra 84.689187 --dec -69.074070 --radius=1.0  --scale-units arcsecperpix --scale-low 0.21 --scale-high 0.26 --overwrite --crpix-x -2284 --crpix-y  -177 --no-tweak --no-plots --new-fits WIN1.CHIP4.OUT1_newwcs.fits WIN1.CHIP4.OUT1.fits
run_command: cmdfile: solvefield_command_WIN1.CHIP4.OUT1_solve_field.txt
run_command: logfile: solvefield_command_WIN1.CHIP4.OUT1_solve_field.log


2024-08-23 15:25:05,741:INFO:create_wcs_frame: creating /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:46:20.918_pre/WFI.2002-03-11T00:46:20.918_pre_wcs.fits
2024-08-23 15:25:06,117:INFO:solving image 02/03: WFI.2002-03-11T00:40:33.523_pre.fits
2024-08-23 15:25:06,118:INFO:


Solving field WFI.2002-03-11T00:40:33.523_pre
2024-08-23 15:25:06,120:INFO:run_solve_field: directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:40:33.523_pre created 
2024-08-23 15:25:06,127:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:40:33.523_pre/WIN1.CHIP1.OUT1_solve_field/ Created 
2024-08-23 15:25:06,243:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_V_89_ESO843/2002-03-11T00:20:42/WFI.2002-03-11T00:40:33.523_pre/WIN1.CHIP2.OUT1_solve_f

...

2024-08-23 15:28:02,058:INFO:solving image 02/03: WFI.2002-03-11T00:24:27.386_pre.fits
2024-08-23 15:28:02,059:INFO:


Solving field WFI.2002-03-11T00:24:27.386_pre
2024-08-23 15:28:02,060:INFO:run_solve_field: directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre created 
2024-08-23 15:28:02,067:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre/WIN1.CHIP1.OUT1_solve_field/ Created 
2024-08-23 15:28:02,114:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre/WIN1.CHIP2.OUT1_solve_field/ Created 
2024-08-23 15:28:02,150:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre/WIN1.CHIP3.OUT

This will be executed when the package is imported.



****************** solvefield ***************
dir_name2: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre/WIN1.CHIP8.OUT1_solve_field  
file_name2: WIN1.CHIP8.OUT1.fits 
sub_dir_name: WIN1.CHIP8.OUT1_solve_field
solvefield: running solve-field in directory: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:24:27.386_pre/WIN1.CHIP8.OUT1_solve_field
******************



solvefield: command: /usr/local/opt/astrometry-net/bin/solve-field  --ra 84.666317 --dec -69.083140 --radius=1.0  --scale-units arcsecperpix --scale-low 0.21 --scale-high 0.26 --overwrite --crpix-x -2274 --crpix-y  3971 --no-tweak --no-plots --new-fits WIN1.CHIP8.OUT1_newwcs.fits WIN1.CHIP8.OUT1.fits
run_command: cmdfile: solvefield_command_WIN1.CHIP8.OUT1_solve_field.txt
run_command: logfile: solvefield_command_WIN1.CHIP8.OUT1_solve_field.log


2024-08-23 15:29:09,565:INFO:solving image 03/03: WFI.2002-03-11T00:30:13.361_pre.fits
2024-08-23 15:29:09,566:INFO:


Solving field WFI.2002-03-11T00:30:13.361_pre
2024-08-23 15:29:09,568:INFO:run_solve_field: directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre created 
2024-08-23 15:29:09,574:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre/WIN1.CHIP1.OUT1_solve_field/ Created 
2024-08-23 15:29:09,701:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre/WIN1.CHIP2.OUT1_solve_field/ Created 
2024-08-23 15:29:09,807:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre/WIN1.CHIP3.OUT

This will be executed when the package is imported.



****************** solvefield ***************
dir_name2: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre/WIN1.CHIP7.OUT1_solve_field  
file_name2: WIN1.CHIP7.OUT1.fits 
sub_dir_name: WIN1.CHIP7.OUT1_solve_field
solvefield: running solve-field in directory: /Users/fselman/Pictures/WFI/30Doradus/science/LMC01/BB_B_99_ESO842/2002-03-11T00:20:42/WFI.2002-03-11T00:30:13.361_pre/WIN1.CHIP7.OUT1_solve_field
******************



solvefield: command: /usr/local/opt/astrometry-net/bin/solve-field  --ra 84.688962 --dec -69.074240 --radius=1.0  --scale-units arcsecperpix --scale-low 0.21 --scale-high 0.26 --overwrite --crpix-x  -135 --crpix-y  3967 --no-tweak --no-plots --new-fits WIN1.CHIP7.OUT1_newwcs.fits WIN1.CHIP7.OUT1.fits
run_command: cmdfile: solvefield_command_WIN1.CHIP7.OUT1_solve_field.txt
run_command: logfile: solvefield_command_WIN1.CHIP7.OUT1_solve_field.log


2024-08-23 15:30:12,705:INFO:

 ****** Py4 Done! ******


